# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`/work/data`)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Les services

| Adresse | Service |
|---|---|
| http://localhost:8888 | JupyterLab |
| http://localhost:8080 | Master Spark |
| http://localhost:4040 | Spark UI de la session ouverte |
| http://localhost:18080 | Spark History |
| `mongodb://mongo:27017` | MongoDB |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, BooleanType, TimestampType
)

# 1. Schéma des produits
schema_produits = StructType([
    StructField("produit_id", StringType(), True),
    StructField("nom", StringType(), True),
    StructField("categorie", StringType(), True),
    StructField("sous_categorie", StringType(), True),
    StructField("prix", DoubleType(), True),
    StructField("poids_kg", DoubleType(), True)
])

# 2. Schéma des commandes
schema_commandes = StructType([
    StructField("commande_id", StringType(), True),
    StructField("client_id", StringType(), True),
    StructField("horodatage", TimestampType(), True),
    StructField("evenement", StringType(), True),
    StructField("livreur_id", StringType(), True),
    StructField("detail", StringType(), True)
])

# 3. Schéma des communes
schema_communes = StructType([
    StructField("commune", StringType(), True),
    StructField("code_postal", StringType(), True),
    StructField("lon", DoubleType(), True),
    StructField("lat", DoubleType(), True)
])

# 4. Schéma des fichiers GPS
schema_gps = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("livreur_id", StringType(), True),
    StructField("lon", DoubleType(), True),
    StructField("lat", DoubleType(), True),
    StructField("vitesse_kmh", DoubleType(), True),
    StructField("dans_vehicule", BooleanType(), True),
    StructField("charge_kg", DoubleType(), True)
])

# Chargement des fichiers CSV
produits = spark.read.csv(
    "/work/data/produits.csv",
    header=True,
    schema=schema_produits
)

commandes = spark.read.csv(
    "/work/data/commandes.csv",
    header=True,
    schema=schema_commandes
)

communes = spark.read.csv(
    "/work/data/communes.csv",
    header=True,
    schema=schema_communes
)

gps = spark.read.csv(
    "/work/data/vehicules/*/*.csv",
    header=True,
    schema=schema_gps
)

# Affichage du nombre de lignes
print("Produits :", produits.count())
print("Commandes :", commandes.count())
print("Communes :", communes.count())
print("GPS :", gps.count())

Produits : 233


Commandes : 846535
Communes : 142


GPS : 5174905


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [7]:
from pyspark.sql import functions as F

# Ajouter l'identifiant du véhicule à partir du chemin
gps = gps.withColumn(
    "vehicule_id",
    F.regexp_extract(
        F.input_file_name(),
        r"vehicules/([^/]+)/",
        1
    )
)

# Extraire la date du point GPS
gps = gps.withColumn(
    "jour",
    F.to_date(F.col("timestamp"))
)

# Afficher le résultat
gps.select(
    "timestamp",
    "livreur_id",
    "vehicule_id",
    "jour"
).show(10, truncate=False)

+-------------------+----------+-----------+----------+
|timestamp          |livreur_id|vehicule_id|jour      |
+-------------------+----------+-----------+----------+
|2025-10-01 07:40:00|L001      |V18        |2025-10-01|
|2025-10-01 07:41:00|L001      |V18        |2025-10-01|
|2025-10-01 07:42:00|L001      |V18        |2025-10-01|
|2025-10-01 07:43:00|L001      |V18        |2025-10-01|
|2025-10-01 07:44:00|L001      |V18        |2025-10-01|
|2025-10-01 07:45:00|L001      |V18        |2025-10-01|
|2025-10-01 07:46:00|L001      |V18        |2025-10-01|
|2025-10-01 07:47:00|L001      |V18        |2025-10-01|
|2025-10-01 07:48:00|L001      |V18        |2025-10-01|
|2025-10-01 07:49:00|L001      |V18        |2025-10-01|
+-------------------+----------+-----------+----------+
only showing top 10 rows



In [8]:
gps.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- livreur_id: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- vitesse_kmh: double (nullable = true)
 |-- dans_vehicule: boolean (nullable = true)
 |-- charge_kg: double (nullable = true)
 |-- vehicule_id: string (nullable = false)
 |-- jour: date (nullable = true)



## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [9]:
# Nombre de commandes distinctes
nb_commandes = commandes.select("commande_id").distinct().count()

# Nombre de clients distincts
nb_clients = commandes.select("client_id").distinct().count()

# Nombre de livreurs distincts
nb_livreurs = commandes.filter(
    F.col("livreur_id").isNotNull()
).select("livreur_id").distinct().count()

# Nombre de véhicules distincts
nb_vehicules = gps.filter(
    F.col("vehicule_id") != ""
).select("vehicule_id").distinct().count()

# Affichage des résultats
print("Nombre de commandes :", nb_commandes)
print("Nombre de clients :", nb_clients)
print("Nombre de livreurs :", nb_livreurs)
print("Nombre de véhicules :", nb_vehicules)

Nombre de commandes : 271617
Nombre de clients : 8000
Nombre de livreurs : 90
Nombre de véhicules : 72


2.2) Comptez les événements du journal par type.

In [10]:
# Nombre d'événements par type

evenements_par_type = (
    commandes
    .groupBy("evenement")
    .count()
    .orderBy(F.desc("count"))
)

# Afficher les résultats
evenements_par_type.show(truncate=False)

+---------+------+
|evenement|count |
+---------+------+
|chargee  |286862|
|commande |271617|
|livree   |268682|
|echec    |11220 |
|reportee |6960  |
|retour   |1194  |
+---------+------+



In [11]:
# Vérifier le nombre total d'événements
evenements_par_type.agg(
    F.sum("count").alias("total_evenements")
).show()

+----------------+
|total_evenements|
+----------------+
|          846535|
+----------------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [12]:
from pyspark.sql import functions as F

# 1. Compter les chargements pour chaque commande
chargements_par_commande = (
    commandes
    .groupBy("commande_id")
    .agg(
        F.sum(
            F.when(F.col("evenement") == "chargee", 1)
             .otherwise(0)
        ).alias("nb_chargements")
    )
)

# 2. Compter les commandes selon le nombre de chargements
repartition_chargements = (
    chargements_par_commande
    .groupBy("nb_chargements")
    .count()
    .withColumnRenamed("count", "nb_commandes")
    .orderBy("nb_chargements")
)

# 3. Afficher les résultats
repartition_chargements.show(truncate=False)

+--------------+------------+
|nb_chargements|nb_commandes|
+--------------+------------+
|0             |1637        |
|1             |254229      |
|2             |14673       |
|3             |1026        |
|4             |51          |
|5             |1           |
+--------------+------------+



In [13]:
# Vérifier que toutes les commandes ont été prises en compte
repartition_chargements.agg(
    F.sum("nb_commandes").alias("total_commandes")
).show()

+---------------+
|total_commandes|
+---------------+
|         271617|
+---------------+



2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [14]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Définir une fenêtre par commande, dans l'ordre chronologique
fenetre = Window.partitionBy("commande_id").orderBy("horodatage")

# 2. Ajouter l'événement précédent
commandes_precedentes = commandes.withColumn(
    "evenement_precedent",
    F.lag("evenement").over(fenetre)
)

# 3. Sélectionner uniquement les événements "chargee"
chargements = commandes_precedentes.filter(
    F.col("evenement") == "chargee"
)

# 4. Compter les chargements par événement précédent
resultat_24 = (
    chargements
    .groupBy("evenement_precedent")
    .count()
    .orderBy(F.desc("count"))
)

# 5. Afficher les résultats
resultat_24.show(truncate=False)

+-------------------+------+
|evenement_precedent|count |
+-------------------+------+
|commande           |269980|
|echec              |9950  |
|reportee           |6932  |
+-------------------+------+



*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [15]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Conserver les événements jusqu'au 31/12/2025 inclus
commandes_2025 = commandes.filter(
    F.col("horodatage") < F.lit("2026-01-01").cast("timestamp")
)

# 2. Définir une fenêtre pour retrouver le dernier événement
fenetre_dernier = (
    Window
    .partitionBy("commande_id")
    .orderBy(F.col("horodatage").desc())
)

# 3. Récupérer le dernier événement de chaque commande
dernier_evenement = (
    commandes_2025
    .withColumn("rang", F.row_number().over(fenetre_dernier))
    .filter(F.col("rang") == 1)
)

# 4. Classer les commandes selon leur situation
situation_commandes = (
    dernier_evenement
    .withColumn(
        "situation",
        F.when(F.col("evenement") == "livree", "Livree")
         .when(F.col("evenement") == "retour", "Abandonnee")
         .otherwise("En cours")
    )
)

# 5. Compter les commandes par situation
resultat_25 = (
    situation_commandes
    .groupBy("situation")
    .count()
    .orderBy("situation")
)

resultat_25.show(truncate=False)

+----------+------+
|situation |count |
+----------+------+
|Abandonnee|1194  |
|En cours  |1741  |
|Livree    |268682|
+----------+------+



In [16]:
# Commandes non livrées au 31/12/2025
non_livrees = situation_commandes.filter(
    F.col("situation").isin("Abandonnee", "En cours")
)

print(
    "Nombre de commandes non livrées :",
    non_livrees.count()
)

Nombre de commandes non livrées : 2935


## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [33]:
from pyspark.sql import functions as F

# 1. Sélectionner les événements de création des commandes
commandes_initiales = commandes.filter(
    F.col("evenement") == "commande"
)

# 2. Séparer les produits et créer une ligne par produit
lignes = (
    commandes_initiales
    .withColumn(
        "produit",
        F.explode(F.split(F.col("detail"), r"\|"))
    )

    # 3. Extraire l'identifiant du produit
    .withColumn(
        "produit_id",
        F.regexp_extract(
            F.col("produit"),
            r"^(P\d+)x\d+$",
            1
        )
    )

    # 4. Extraire la quantité
    .withColumn(
        "quantite",
        F.regexp_extract(
            F.col("produit"),
            r"x(\d+)$",
            1
        ).cast("int")
    )

    # 5. Conserver uniquement les colonnes demandées
    .select(
        "commande_id",
        "client_id",
        "horodatage",
        "produit_id",
        "quantite"
    )
)

# 6. Afficher les résultats
lignes.show(30, truncate=False)

+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|horodatage         |produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|C0000001   |CL05658  |2025-01-01 00:03:44|P0037     |5       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0063     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0019     |2       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0067     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0069     |2       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0089     |3       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0061     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0055     |2       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0009     |3       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0076     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0107     |1       |
|C0000001   |CL05658  |2025-01-01 00:03:44|P0005     |2       |
|C0000001   |CL05658  |2025-01-01 00:03:

In [34]:
# Nombre total de lignes de commande
print("Nombre de lignes :", lignes.count())

# Vérifier les types des colonnes
lignes.printSchema()

# Vérifier les erreurs d'extraction
lignes.filter(
    (F.col("produit_id") == "") |
    F.col("quantite").isNull()
).count()

Nombre de lignes : 4779049
root
 |-- commande_id: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- horodatage: timestamp (nullable = true)
 |-- produit_id: string (nullable = false)
 |-- quantite: integer (nullable = true)



0

3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [42]:
from pyspark.sql import functions as F

# =====================================================
# QUESTION 3.2 : LIGNES ET MONTANTS DES COMMANDES
# =====================================================

# 1. Ajouter les informations du catalogue
# et calculer le montant et le poids de chaque ligne

lignes = (
    lignes
    .join(produits, on="produit_id", how="left")

    # Calculer le montant de chaque ligne
    .withColumn(
        "montant",
        F.col("prix") * F.col("quantite")
    )

    # Calculer le poids total de chaque ligne
    .withColumn(
        "poids_total_kg",
        F.col("poids_kg") * F.col("quantite")
    )
)

# 2. Construire une table avec une ligne par commande

commandes_resume = (
    lignes
    .groupBy("commande_id", "client_id")
    .agg(
        F.sum("montant").alias("montant"),
        F.sum("poids_total_kg").alias("poids_kg"),
        F.count("*").alias("nb_lignes")
    )
)

# 3. Afficher les lignes de commande
# Montants et poids avec exactement 2 décimales

print("TABLE DES LIGNES DE COMMANDE")

lignes.select(
    "commande_id",
    "produit_id",
    "quantite",
    F.format_number("prix", 2).alias("prix"),
    F.format_number("montant", 2).alias("montant"),
    F.format_number("poids_total_kg", 2).alias("poids_total_kg")
).show(10, truncate=False)


# 4. Afficher le récapitulatif des commandes
# Montants et poids avec exactement 2 décimales

print("TABLE RÉCAPITULATIVE DES COMMANDES")

commandes_resume.select(
    "commande_id",
    "client_id",
    F.format_number("montant", 2).alias("montant"),
    F.format_number("poids_kg", 2).alias("poids_kg"),
    "nb_lignes"
).show(10, truncate=False)


# =====================================================
# VÉRIFICATION DES RÉSULTATS
# =====================================================

print("Nombre de commandes :", commandes_resume.count())

print(
    "Nombre total de lignes :",
    commandes_resume.agg(
        F.sum("nb_lignes")
    ).first()[0]
)

print(
    "Produits non trouvés dans le catalogue :",
    lignes.filter(F.col("prix").isNull()).count()
)

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `prix` is ambiguous, could be: [`prix`, `prix`].

In [ ]:
# Vérifier le nombre total de commandes
print("Nombre de commandes :", commandes_resume.count())

# Vérifier le nombre total de lignes
print(
    "Nombre total de lignes :",
    commandes_resume.agg(
        F.sum("nb_lignes")
    ).first()[0]
)

# Vérifier les produits absents du catalogue
print(
    "Produits non trouvés dans le catalogue :",
    lignes.filter(F.col("prix").isNull()).count()
)

Nombre de commandes : 271617


Nombre total de lignes : 4779049


Produits non trouvés dans le catalogue : 0


## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Définir les fenêtres chronologiques
fenetre_debut = (
    Window.partitionBy("vehicule_id", "jour")
    .orderBy(F.col("timestamp").asc())
)

fenetre_fin = (
    Window.partitionBy("vehicule_id", "jour")
    .orderBy(F.col("timestamp").desc())
)

# 2. Identifier le premier et le dernier point GPS
gps_points = (
    gps
    .withColumn("rang_debut", F.row_number().over(fenetre_debut))
    .withColumn("rang_fin", F.row_number().over(fenetre_fin))
)

# 3. Conserver les points de début et de fin
points_entrepots = (
    gps_points
    .filter(
        (F.col("rang_debut") == 1) |
        (F.col("rang_fin") == 1)
    )
    .select(
        "vehicule_id",
        "jour",
        "lon",
        "lat"
    )
)

# 4. Arrondir les coordonnées à 0,01 degré
points_entrepots = (
    points_entrepots
    .withColumn("lon_arrondie", F.round("lon", 2))
    .withColumn("lat_arrondie", F.round("lat", 2))
)

# 5. Compter les journées par position
positions_frequentes = (
    points_entrepots
    .groupBy("lon_arrondie", "lat_arrondie")
    .agg(
        F.count("*").alias("nb_points"),
        F.countDistinct("jour").alias("nb_jours")
    )
    .orderBy(F.desc("nb_points"))
)

# 6. Afficher les positions les plus fréquentes
positions_frequentes.show(20, truncate=False)

+------------+------------+---------+--------+
|lon_arrondie|lat_arrondie|nb_points|nb_jours|
+------------+------------+---------+--------+
|2.45        |48.91       |8434     |302     |
|2.28        |48.94       |6416     |302     |
|2.35        |48.75       |5934     |302     |
+------------+------------+---------+--------+



In [ ]:
# Nombre de journées-véhicules
nb_journees_vehicules = (
    gps
    .select("vehicule_id", "jour")
    .distinct()
    .count()
)

print("Nombre de journées-véhicules :", nb_journees_vehicules)

# Nombre de positions GPS arrondies différentes
print(
    "Nombre de positions différentes :",
    positions_frequentes.count()
)

Nombre de journées-véhicules : 10392


Nombre de positions différentes : 3


4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [40]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Récupérer le premier point GPS de chaque véhicule et chaque jour
fenetre_premier = (
    Window.partitionBy("vehicule_id", "jour")
    .orderBy(F.col("timestamp").asc())
)

premiers_points = (
    gps
    .withColumn("rang", F.row_number().over(fenetre_premier))
    .filter(F.col("rang") == 1)
    .withColumn("lon_arrondie", F.round("lon", 2))
    .withColumn("lat_arrondie", F.round("lat", 2))
)

# 2. Calculer la position moyenne de chaque entrepôt
entrepots_positions = (
    premiers_points
    .groupBy("lon_arrondie", "lat_arrondie")
    .agg(
        F.avg("lon").alias("lon"),
        F.avg("lat").alias("lat")
    )
)

# 3. Numéroter les entrepôts par longitude croissante
fenetre_entrepots = Window.orderBy(F.col("lon").asc())

entrepots = (
    entrepots_positions
    .withColumn(
        "entrepot",
        F.concat(
            F.lit("E"),
            F.row_number().over(fenetre_entrepots)
        )
    )
    .select("entrepot", "lon", "lat")
    .orderBy("entrepot")
)

print("TABLE DES ENTREPÔTS")
entrepots.show(truncate=False)

# 4. Associer chaque véhicule à son entrepôt
vehicule_entrepot = (
    premiers_points
    .groupBy("vehicule_id", "lon_arrondie", "lat_arrondie")
    .count()
    .withColumn(
        "rang",
        F.row_number().over(
            Window.partitionBy("vehicule_id")
            .orderBy(F.desc("count"))
        )
    )
    .filter(F.col("rang") == 1)
    .join(
        entrepots_positions
        .select("lon_arrondie", "lat_arrondie", "lon"),
        on=["lon_arrondie", "lat_arrondie"],
        how="inner"
    )
    .join(
        entrepots.select("entrepot", "lon"),
        on="lon",
        how="inner"
    )
    .select("vehicule_id", "entrepot")
)

print("TABLE VÉHICULE - ENTREPÔT")
vehicule_entrepot.show(20, truncate=False)

# 5. Compter les véhicules par entrepôt
vehicules_par_entrepot = (
    vehicule_entrepot
    .groupBy("entrepot")
    .agg(
        F.countDistinct("vehicule_id").alias("nb_vehicules")
    )
    .orderBy("entrepot")
)

print("NOMBRE DE VÉHICULES PAR ENTREPÔT")
vehicules_par_entrepot.show()

TABLE DES ENTREPÔTS


15:11:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:11:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:11:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:26 WARN WindowExec: No Partition Defined for Window operat

+--------+------------------+------------------+
|entrepot|lon               |lat               |
+--------+------------------+------------------+
|E1      |2.277068913341649 |48.936712904613536|
|E2      |2.351513594202888 |48.747046985844335|
|E3      |2.4489591059995286|48.906701645482606|
+--------+------------------+------------------+

TABLE VÉHICULE - ENTREPÔT


15:12:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:53 WARN WindowExec: No Partition Defined for Window operat

+-----------+--------+
|vehicule_id|entrepot|
+-----------+--------+
|V01        |E1      |
|V02        |E1      |
|V03        |E1      |
|V04        |E1      |
|V05        |E1      |
|V06        |E1      |
|V07        |E1      |
|V08        |E1      |
|V09        |E1      |
|V10        |E1      |
|V11        |E1      |
|V12        |E1      |
|V13        |E1      |
|V14        |E1      |
|V15        |E1      |
|V16        |E1      |
|V17        |E1      |
|V18        |E1      |
|V19        |E2      |
|V20        |E2      |
+-----------+--------+
only showing top 20 rows

NOMBRE DE VÉHICULES PAR ENTREPÔT


15:12:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:12:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:13:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:13:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:13:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:13:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:13:22 WARN WindowExec: No Partition Defined for Window operat

+--------+------------+
|entrepot|nb_vehicules|
+--------+------------+
|      E1|          24|
|      E2|          24|
|      E3|          24|
+--------+------------+



In [41]:
print("Nombre d'entrepôts :", entrepots.count())

print(
    "Nombre de véhicules affectés :",
    vehicule_entrepot.select("vehicule_id").distinct().count()
)

print(
    "Total véhicules :",
    vehicules_par_entrepot.agg(
        F.sum("nb_vehicules")
    ).first()[0]
)

Nombre d'entrepôts : 3


Nombre de véhicules affectés : 72


15:14:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:14:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:14:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:14:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:14:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:14:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:14:52 WARN WindowExec: No Partition Defined for Window operat

Total véhicules : 72


## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

In [43]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =====================================================
# QUESTION 5.1 : IDENTIFIER LES DEBUTS DE LIVRAISON
# =====================================================

# 1. Définir une fenêtre par livreur et par jour
fenetre_livreur = (
    Window
    .partitionBy("livreur_id", "jour")
    .orderBy("timestamp")
)

# 2. Récupérer l'état précédent du livreur
gps = gps.withColumn(
    "etat_precedent",
    F.lag("dans_vehicule").over(fenetre_livreur)
)

# 3. Ajouter la colonne chargement
gps = gps.withColumn(
    "chargement",
    F.when(
        (F.col("etat_precedent") == True) &
        (F.col("dans_vehicule") == False),
        1
    ).otherwise(0)
)

# 4. Afficher les chargements du livreur L001
# pour le 6 janvier 2025

resultat_51 = (
    gps
    .filter(
        (F.col("livreur_id") == "L001") &
        (F.col("jour") == F.lit("2025-01-06").cast("date")) &
        (F.col("chargement") == 1)
    )
    .select(
        "timestamp",
        "livreur_id",
        "vehicule_id",
        "dans_vehicule",
        "chargement"
    )
    .orderBy("timestamp")
)

print("DEBUTS DE LIVRAISON - L001 - 2025-01-06")

resultat_51.show(100, truncate=False)

DEBUTS DE LIVRAISON - L001 - 2025-01-06


+-------------------+----------+-----------+-------------+----------+
|timestamp          |livreur_id|vehicule_id|dans_vehicule|chargement|
+-------------------+----------+-----------+-------------+----------+
|2025-01-06 09:54:00|L001      |V10        |false        |1         |
|2025-01-06 10:00:00|L001      |V10        |false        |1         |
|2025-01-06 10:09:00|L001      |V10        |false        |1         |
|2025-01-06 10:23:00|L001      |V10        |false        |1         |
|2025-01-06 10:36:00|L001      |V10        |false        |1         |
|2025-01-06 10:54:00|L001      |V10        |false        |1         |
|2025-01-06 11:12:00|L001      |V10        |false        |1         |
|2025-01-06 11:38:00|L001      |V10        |false        |1         |
|2025-01-06 12:02:00|L001      |V10        |false        |1         |
|2025-01-06 12:15:00|L001      |V10        |false        |1         |
|2025-01-06 13:04:00|L001      |V10        |false        |1         |
|2025-01-06 13:42:00

In [44]:
print(
    "Nombre de débuts de livraison pour L001 :",
    resultat_51.count()
)

gps.select(
    "timestamp",
    "livreur_id",
    "jour",
    "dans_vehicule",
    "etat_precedent",
    "chargement"
).filter(
    (F.col("livreur_id") == "L001") &
    (F.col("jour") == F.lit("2025-01-06").cast("date"))
).orderBy("timestamp").show(30, truncate=False)

Nombre de débuts de livraison pour L001 : 16


+-------------------+----------+----------+-------------+--------------+----------+
|timestamp          |livreur_id|jour      |dans_vehicule|etat_precedent|chargement|
+-------------------+----------+----------+-------------+--------------+----------+
|2025-01-06 07:43:00|L001      |2025-01-06|false        |NULL          |0         |
|2025-01-06 07:44:00|L001      |2025-01-06|false        |false         |0         |
|2025-01-06 07:45:00|L001      |2025-01-06|false        |false         |0         |
|2025-01-06 07:46:00|L001      |2025-01-06|false        |false         |0         |
|2025-01-06 07:47:00|L001      |2025-01-06|false        |false         |0         |
|2025-01-06 07:48:00|L001      |2025-01-06|false        |false         |0         |
|2025-01-06 07:49:00|L001      |2025-01-06|false        |false         |0         |
|2025-01-06 07:50:00|L001      |2025-01-06|false        |false         |0         |
|2025-01-06 07:51:00|L001      |2025-01-06|false        |false         |0   

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

In [45]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =====================================================
# QUESTION 5.2 : NUMEROTER LES LIVRAISONS
# =====================================================

# 1. Définir la fenêtre chronologique par livreur et par jour
fenetre_cumul = (
    Window
    .partitionBy("livreur_id", "jour")
    .orderBy("timestamp")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# 2. Calculer le numéro de livraison
gps = gps.withColumn(
    "numero_livraison",
    F.sum("chargement").over(fenetre_cumul)
)

# 3. Sélectionner le livreur L001 le 06/01/2025
gps_L001 = gps.filter(
    (F.col("livreur_id") == "L001") &
    (F.col("jour") == F.lit("2025-01-06").cast("date"))
)

# 4. Afficher les périodes par numéro de livraison
resultat_52 = (
    gps_L001
    .groupBy("numero_livraison")
    .agg(
        F.min("timestamp").alias("premier_timestamp"),
        F.max("timestamp").alias("dernier_timestamp"),
        F.sum(
            F.when(F.col("dans_vehicule") == False, 1).otherwise(0)
        ).alias("minutes_a_pied")
    )
    .orderBy("numero_livraison")
)

print("LIVRAISONS DU LIVREUR L001 - 06/01/2025")

resultat_52.show(100, truncate=False)

LIVRAISONS DU LIVREUR L001 - 06/01/2025


+----------------+-------------------+-------------------+--------------+
|numero_livraison|premier_timestamp  |dernier_timestamp  |minutes_a_pied|
+----------------+-------------------+-------------------+--------------+
|0               |2025-01-06 07:43:00|2025-01-06 09:53:00|44            |
|1               |2025-01-06 09:54:00|2025-01-06 09:59:00|4             |
|2               |2025-01-06 10:00:00|2025-01-06 10:08:00|7             |
|3               |2025-01-06 10:09:00|2025-01-06 10:22:00|10            |
|4               |2025-01-06 10:23:00|2025-01-06 10:35:00|4             |
|5               |2025-01-06 10:36:00|2025-01-06 10:53:00|8             |
|6               |2025-01-06 10:54:00|2025-01-06 11:11:00|4             |
|7               |2025-01-06 11:12:00|2025-01-06 11:37:00|18            |
|8               |2025-01-06 11:38:00|2025-01-06 12:01:00|17            |
|9               |2025-01-06 12:02:00|2025-01-06 12:14:00|4             |
|10              |2025-01-06 12:15:00|

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

7.2) Panier moyen : montant moyen d'une commande.

7.3) Panier médian.

7.4) Top 10 des produits par chiffre d'affaires.

7.5) Top 10 des produits par quantité vendue.

7.6) Top 10 des clients par dépense sur l'année.

7.7) Top 10 des clients par nombre de commandes.

## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

8.4) Motifs d'échec : nombre d'échecs par motif.

8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*